<a href="https://colab.research.google.com/github/Sarah-0405/Cold_Spots_Bayern/blob/main/Erreichbarkeitsanalyse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

input Daten:

- OSMnx Graph Daten für jede Stadt
- Startpunkte für Service Area Analyse

Output:

reachable edges => = Service Area = Straßenabschnitte, die in x minuten erreichbar sind (müssen dann noch zu Polygonen umgewandelt werden, das mache ich dann)


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.5/101.5 kB 3.8 MB/s eta 0:00:00


In [3]:
import osmnx as ox
import os
import geopandas as gpd
import pandas as pd
import networkx as nx

In [4]:
# 1. Wegegraphen laden

# Define the directory where the projected graphml files are saved in Google Drive
graph_data_input = "/content/drive/MyDrive/Cold Spots Bayern/OSMnx_Graphen_proj" # Pfad anpassen

# Initialize a dictionary to store the loaded projected graphs
graphs_proj = {}

print(f"Loading projected graphs from: {graph_data_input}")

# Load the projected street networks from the graphml files
if os.path.exists(graph_data_input):
    for filename in os.listdir(graph_data_input):
        if filename.endswith("_walk_proj.graphml"):
            # Extract city name from filename
            city_name = filename.replace("_walk_proj.graphml", "").replace("_", " ")
            # Handle specific city name variations if necessary to match original keys if needed later
            city_name = city_name.replace("Fuerth", "Fürth").replace("Wuerzburg", "Würzburg").replace("Kempten Allgaeu", "Kempten (Allgäu)")

            filepath = os.path.join(graph_data_input, filename)
            try:
                G_proj = ox.load_graphml(filepath)
                graphs_proj[city_name] = G_proj
                print(f"Loaded projected street network for {city_name}")
            except Exception as e:
                print(f"Could not load projected street network for {city_name}: {e}")
else:
    print(f"Directory not found: {graph_data_input}")

print("\nFinished loading projected street networks.")


Loading projected graphs from: /content/drive/MyDrive/Cold Spots Bayern/OSMnx_Graphen_proj
Loaded projected street network for Aschaffenburg
Loaded projected street network for Augsburg
Loaded projected street network for Bamberg
Loaded projected street network for Bayreuth
Loaded projected street network for Erlangen
Loaded projected street network for Fürth
Loaded projected street network for Ingolstadt
Loaded projected street network for Kempten (Allgäu)
Loaded projected street network for Landshut
Loaded projected street network for Munich
Loaded projected street network for Nuremberg
Loaded projected street network for Passau
Loaded projected street network for Regensburg
Loaded projected street network for Rosenheim
Loaded projected street network for Schweinfurt
Loaded projected street network for Würzburg

Finished loading projected street networks.


Startpunkte nacheinander laden mit Check

In [ ]:
import geopandas as gpd
import os

starting_points_input = "/content/drive/MyDrive/Cold Spots Bayern/all_starting_points_2019_2024_neu"

for file in os.listdir(starting_points_input):
    if file.endswith(".gpkg"):
        file_path = os.path.join(starting_points_input, file)
        try:
            gdf = gpd.read_file(file_path)
            print(f"OK: {file}, {len(gdf)} Features")
        except Exception as e:
            print(f"Fehler bei {file}: {e}")


OK: Aschaffenburg_starting_points_2019_2024.gpkg, 68603 Features
OK: Augsburg_starting_points_2019_2024.gpkg, 171991 Features
OK: Bamberg_starting_points_2019_2024.gpkg, 144623 Features
OK: Bayreuth_starting_points_2019_2024.gpkg, 169366 Features
OK: Erlangen_starting_points_2019_2024.gpkg, 146305 Features
OK: Fuerth_starting_points_2019_2024.gpkg, 110152 Features
OK: Ingolstadt_starting_points_2019_2024.gpkg, 116108 Features
OK: Kempten_Allgaeu_starting_points_2019_2024.gpkg, 60799 Features
OK: Landshut_starting_points_2019_2024.gpkg, 152307 Features
OK: Munich_starting_points_2019_2024.gpkg, 743668 Features
OK: Nuremberg_starting_points_2019_2024.gpkg, 313484 Features
OK: Passau_starting_points_2019_2024.gpkg, 55375 Features
OK: Regensburg_starting_points_2019_2024.gpkg, 217753 Features
OK: Rosenheim_starting_points_2019_2024.gpkg, 55578 Features
OK: Schweinfurt_starting_points_2019_2024.gpkg, 53988 Features
OK: Wuerzburg_starting_points_2019_2024.gpkg, 132280 Features


In [5]:
# 2. Startpunkte laden

# Pfad zu deinem Ordner mit den gpkg-Dateien
starting_points_input = "/content/drive/MyDrive/Cold Spots Bayern/all_starting_points_2019_2024_neu" # Pfad anpassen

all_starting_points_2019_2024 = {}

# Alle Dateien im Ordner durchgehen
for file in os.listdir(starting_points_input):
    if file.endswith(".gpkg"):
        file_path = os.path.join(starting_points_input, file)
        # Name ohne Dateiendung als Key verwenden
        key = os.path.splitext(file)[0]
        # GeoDataFrame laden
        all_starting_points_2019_2024[key] = gpd.read_file(file_path)

# Prüfen, wie viele geladen wurden
print(f"{len(all_starting_points_2019_2024)} Dateien geladen:")
print(all_starting_points_2019_2024.keys())

16 Dateien geladen:
dict_keys(['Landshut_starting_points_2019_2024', 'Erlangen_starting_points_2019_2024', 'Kempten_Allgaeu_starting_points_2019_2024', 'Ingolstadt_starting_points_2019_2024', 'Bayreuth_starting_points_2019_2024', 'Fuerth_starting_points_2019_2024', 'Augsburg_starting_points_2019_2024', 'Aschaffenburg_starting_points_2019_2024', 'Munich_starting_points_2019_2024', 'Bamberg_starting_points_2019_2024', 'Rosenheim_starting_points_2019_2024', 'Regensburg_starting_points_2019_2024', 'Nuremberg_starting_points_2019_2024', 'Passau_starting_points_2019_2024', 'Schweinfurt_starting_points_2019_2024', 'Wuerzburg_starting_points_2019_2024'])


In [6]:
for city, gdf in all_starting_points_2019_2024.items():
    print(f"{city}: {len(gdf)} Starting points")

Landshut_starting_points_2019_2024: 1498 Starting points
Erlangen_starting_points_2019_2024: 2425 Starting points
Kempten_Allgaeu_starting_points_2019_2024: 1159 Starting points
Ingolstadt_starting_points_2019_2024: 3035 Starting points
Bayreuth_starting_points_2019_2024: 1904 Starting points
Fuerth_starting_points_2019_2024: 2311 Starting points
Augsburg_starting_points_2019_2024: 3469 Starting points
Aschaffenburg_starting_points_2019_2024: 1390 Starting points
Munich_starting_points_2019_2024: 13782 Starting points
Bamberg_starting_points_2019_2024: 1559 Starting points
Rosenheim_starting_points_2019_2024: 992 Starting points
Regensburg_starting_points_2019_2024: 2664 Starting points
Nuremberg_starting_points_2019_2024: 5898 Starting points
Passau_starting_points_2019_2024: 1450 Starting points
Schweinfurt_starting_points_2019_2024: 946 Starting points
Wuerzburg_starting_points_2019_2024: 2810 Starting points


In [7]:
# 3. travel speed und travel time definieren

# Define walking speed in meters per second
travel_speed = 1 # m/s

# Define travel times in minutes
travel_times_minutes = [5, 10, 15, 20, 25, 30]

print(f"Walking speed: {travel_speed} m/s")
print(f"Travel times in minutes: {travel_times_minutes}")

Walking speed: 1 m/s
Travel times in minutes: [5, 10, 15, 20, 25, 30]


In [8]:
# 4. travel time für jeden Straßenabschnitt (edge) im Graphen berechnen

# Iterate through the graphs dictionary
for city_name, G in graphs_proj.items():
    print(f"Calculating travel times for edges in {city_name} graph...")

    # Calculate travel time for each edge
    # Ensure the graph has the 'length' attribute for edges
    if all('length' in G.edges[u, v, k] for u, v, k in G.edges):
        # Add 'travel_time' attribute (in seconds) to each edge
        for u, v, k, data in G.edges(keys=True, data=True):
            data['travel_time'] = data['length'] / travel_speed
        print(f"  Added 'travel_time' attribute to edges in {city_name} graph.")
    else:
        print(f"  Warning: 'length' attribute not found on all edges in {city_name} graph. Skipping travel time calculation for this graph.")

print("\nFinished calculating travel times for edges in all graphs.")

# You can inspect the edges of a graph to see the new attribute
# For example: list(graphs['Nuremberg'].edges(data=True))[0]

Calculating travel times for edges in Aschaffenburg graph...
  Added 'travel_time' attribute to edges in Aschaffenburg graph.
Calculating travel times for edges in Augsburg graph...
  Added 'travel_time' attribute to edges in Augsburg graph.
Calculating travel times for edges in Bamberg graph...
  Added 'travel_time' attribute to edges in Bamberg graph.
Calculating travel times for edges in Bayreuth graph...
  Added 'travel_time' attribute to edges in Bayreuth graph.
Calculating travel times for edges in Erlangen graph...
  Added 'travel_time' attribute to edges in Erlangen graph.
Calculating travel times for edges in Fürth graph...
  Added 'travel_time' attribute to edges in Fürth graph.
Calculating travel times for edges in Ingolstadt graph...
  Added 'travel_time' attribute to edges in Ingolstadt graph.
Calculating travel times for edges in Kempten (Allgäu) graph...
  Added 'travel_time' attribute to edges in Kempten (Allgäu) graph.
Calculating travel times for edges in Landshut gra

In [ ]:
# 5. Service Area Analyse

# Dictionary to store reachable nodes and edges for each city, group_id, and travel time
reachable_areas_all_cities = {}

# Convert travel times from minutes to seconds
travel_times_seconds = [t * 60 for t in travel_times_minutes]

# Define the output directory for reachable edges in Google Drive
output_dir_service_area = "/content/drive/MyDrive/Cold Spots Bayern/reachable_edges_2019_2024_Puffer" # evtl anpassen oder so lassen :)

# Create the output directory if it doesn't exist
os.makedirs(output_dir_service_area, exist_ok=True)
print(f"Saving reachable edges to: {output_dir_service_area}")

# Create a mapping from the cleaned city names (used in graphs_proj)
# to the keys in all_starting_points_2019_2024 (the original filenames)
starting_points_key_mapping = {}
if 'all_starting_points_2019_2024' in locals():
    for key in all_starting_points_2019_2024.keys():
        # Extract city name from the starting_points_2019_2024 key (e.g., "Aschaffenburg_starting_points_2019_2024" -> "Aschaffenburg")
        # Handle potential variations in naming (e.g., umlauts, spaces) to match graphs_proj keys
        cleaned_city_name = key.replace("_starting_points_2019_2024", "").replace("_", " ")
        # Further refine cleaning to match graphs_proj keys if necessary
        cleaned_city_name = cleaned_city_name.replace("Fuerth", "Fürth").replace("Wuerzburg", "Würzburg").replace("Kempten Allgaeu", "Kempten (Allgäu)")
        starting_points_key_mapping[cleaned_city_name] = key

# Ask the user for the city to start with
start_city = input("Please enter the city name to start processing with (e.g., 'Nuremberg'): ")

# Create a list of cities to process, starting with the specified city
cities_to_process = [start_city] + [city for city in graphs_proj.keys() if city != start_city]

# Iterate through the cities
for city_name in cities_to_process:
     # Check if the city exists in the graphs_proj keys
    if city_name not in graphs_proj.keys():
        print(f"\nSkipping city: {city_name}. Graph data not found for this city.")
        continue

    print(f"\nCalculating reachable nodes and edges for city: {city_name}")

    # Initialize dictionary for the current city
    reachable_areas_all_cities[city_name] = {}

    # Access the street network graph for the current city
    # The graph is already projected and has travel times calculated
    G = graphs_proj[city_name]

    # Find the corresponding starting points GeoDataFrame for the current city using the mapping.
    if city_name in starting_points_key_mapping:
        starting_points_key = starting_points_key_mapping[city_name]
        if starting_points_key in all_starting_points_2019_2024 and not all_starting_points_2019_2024[starting_points_key].empty:
            starting_point_gdf = all_starting_points_2019_2024[starting_points_key]
            print(f"  Accessed starting points GeoDataFrame for {city_name} using key: {starting_points_key}")

             # Project the starting points to the graph's CRS if they are different
            if starting_point_gdf.crs != G.graph['crs']:
                print(f"  Projecting starting points for {city_name} to graph CRS: {G.graph['crs']}")
                starting_point_gdf = starting_point_gdf.to_crs(G.graph['crs'])
                print(f"  Finished projecting starting points for {city_name}")
            else:
                print(f"  CRS of starting points matches graph CRS for {city_name}: {G.graph['crs']}")


            # Iterate through each starting point in the city's GeoDataFrame
            print(f"  Processing {len(starting_point_gdf)} starting points for {city_name}...")
            for index, starting_point in starting_point_gdf.iterrows():
                # Get the group_id for this starting point
                group_id = starting_point.get('group_id', 'unknown_group') # Use .get() with a default for safety

                # Initialize dictionary for the current group_id within the city if it doesn't exist
                if group_id not in reachable_areas_all_cities[city_name]:
                     reachable_areas_all_cities[city_name][group_id] = {}

                # Initialize another nested dictionary for the current starting point's index
                if index not in reachable_areas_all_cities[city_name][group_id]:
                    reachable_areas_all_cities[city_name][group_id][index] = {}

                # Find the nearest node in the projected graph to the starting point's geometry
                try:
                    nearest_node_osmid = ox.nearest_nodes(G, starting_point.geometry.x, starting_point.geometry.y)
                except Exception as e:
                    print(f"      Could not find nearest node for starting point index {index} (group_id: {group_id}): {e}. Skipping this point.")
                    continue

                # For each travel time:
                for travel_time_sec, travel_time_min in zip(travel_times_seconds, travel_times_minutes):
                    # print(f"      Calculating reachable nodes for {travel_time_min} minutes for starting point index {index}, group_id {group_id}...") # Removed detailed print
                    try:
                        distances, _ = nx.single_source_dijkstra(G, source=nearest_node_osmid, cutoff=travel_time_sec, weight='travel_time')
                        reachable_nodes_osmid = set(distances.keys())

                        edges_proj_list = list(G.edges(keys=True, data=True))
                        reachable_edges = []
                        for u, v, k, data in edges_proj_list:
                             if u in reachable_nodes_osmid and v in reachable_nodes_osmid:
                                  reachable_edges.append((u, v, k, data))


                        reachable_areas_all_cities[city_name][group_id][index][travel_time_min] = {
                            'reachable_nodes_osmid': reachable_nodes_osmid,
                            'reachable_edges': reachable_edges
                        }
                    except Exception as e:
                        print(f"      Could not calculate reachable nodes/edges for {travel_time_min} minutes for starting point index {index}, group_id {group_id}: {e}. Skipping this travel time for this point.")
                        reachable_areas_all_cities[city_name][group_id][index][travel_time_min] = {
                            'reachable_nodes_osmid': set(),
                            'reachable_edges': []
                        }

            # After processing all starting points for a city, save the reachable edges data
            print(f"\nSaving reachable edges for {city_name}...")
            city_reachable_edges_data = reachable_areas_all_cities[city_name]

            # Prepare data for saving: flatten the structure for easier GeoDataFrame creation
            # Each row in the output GeoDataFrame will represent a reachable edge with its associated
            # city, group_id, starting_point_index, travel_time_min, and geometry.
            edges_to_save = []
            for group_id, group_data in city_reachable_edges_data.items():
                for starting_point_index, travel_time_data in group_data.items():
                    for travel_time_min, reachable_data in travel_time_data.items():
                        for u, v, k, data in reachable_data['reachable_edges']:
                            if 'geometry' in data:
                                edges_to_save.append({
                                    'city': city_name,
                                    'group_id': group_id,
                                    'start_pt_idx': starting_point_index, # Include starting point index for traceability
                                    'travel_time': travel_time_min,
                                    'osmid_u': u, # Include OSM IDs of the edge nodes
                                    'osmid_v': v,
                                    'osmid_k': k, # Include key for MultiDiGraph
                                    'geometry': data['geometry']
                                })

            if edges_to_save:
                # Create a GeoDataFrame from the collected edge data
                # Use the same CRS as the projected graph
                reachable_edges_gdf = gpd.GeoDataFrame(edges_to_save, crs=G.graph['crs'])

                # Create a safe filename for the city
                safe_city_name = city_name.replace(" ", "_").replace("(", "").replace(")", "").replace("̈", "e").replace("ä", "ae").replace("ö", "oe").replace("ü", "ue")
                output_filepath = os.path.join(output_dir_service_area, f"{safe_city_name}_reachable_edges.gpkg")

                try:
                    # Save the GeoDataFrame to a GeoPackage file
                    reachable_edges_gdf.to_file(output_filepath, driver='GPKG')
                    print(f"  Successfully saved reachable edges for {city_name} to {output_filepath}")
                except Exception as e:
                    print(f"  Error saving reachable edges for {city_name} to {output_filepath}: {e}")
            else:
                print(f"  No reachable edges found for {city_name} to save.")


        else:
            print(f"  Starting points GeoDataFrame not found or is empty for {city_name} using key {starting_points_key}. Skipping reachable area calculation for this city.")

    else:
         print(f"  Starting points key not found for {city_name} in the mapping. Skipping reachable area calculation for this city.")


print("\nFinished calculating and saving reachable nodes and edges for all cities.")

Saving reachable edges to: /content/drive/MyDrive/Cold Spots Bayern/reachable_edges_2019_2024_Puffer
Please enter the city name to start processing with (e.g., 'Nuremberg'): Kempten (Allgaeu)

Skipping city: Kempten (Allgaeu). Graph data not found for this city.

Calculating reachable nodes and edges for city: Aschaffenburg
  Accessed starting points GeoDataFrame for Aschaffenburg using key: Aschaffenburg_starting_points_2019_2024
  CRS of starting points matches graph CRS for Aschaffenburg: epsg:32632
  Processing 1390 starting points for Aschaffenburg...


KeyboardInterrupt: 

In [9]:
# 5. Service Area Analyse (Isochronen statt Edges)

import os
import networkx as nx
import osmnx as ox
import geopandas as gpd
from shapely.ops import unary_union, polygonize
from shapely.geometry import MultiLineString

# Dictionary to store isochrones for each city, group_id, and travel time
isochrones_all_cities = {}

# Convert travel times from minutes to seconds
travel_times_seconds = [t * 60 for t in travel_times_minutes]

# Define the output directory for isochrones in Google Drive
output_dir_service_area = "/content/drive/MyDrive/Cold Spots Bayern/isochrones_2019_2024"
os.makedirs(output_dir_service_area, exist_ok=True)
print(f"Saving isochrones to: {output_dir_service_area}")

# Create a mapping from the cleaned city names (used in graphs_proj)
# to the keys in all_starting_points_2019_2024 (the original filenames)
starting_points_key_mapping = {}
if 'all_starting_points_2019_2024' in locals():
    for key in all_starting_points_2019_2024.keys():
        cleaned_city_name = key.replace("_starting_points_2019_2024", "").replace("_", " ")
        cleaned_city_name = (
            cleaned_city_name.replace("Fuerth", "Fürth")
            .replace("Wuerzburg", "Würzburg")
            .replace("Kempten Allgaeu", "Kempten (Allgäu)")
        )
        starting_points_key_mapping[cleaned_city_name] = key

# Ask the user for the city to start with
start_city = input("Please enter the city name to start processing with (e.g., 'Nuremberg'): ")

# Create a list of cities to process, starting with the specified city
cities_to_process = [start_city] + [city for city in graphs_proj.keys() if city != start_city]

# Iterate through the cities
for city_name in cities_to_process:
    if city_name not in graphs_proj.keys():
        print(f"\nSkipping city: {city_name}. Graph data not found for this city.")
        continue

    print(f"\nCalculating isochrones for city: {city_name}")
    isochrones_all_cities[city_name] = {}

    # Access the street network graph for the current city
    G = graphs_proj[city_name]

    # Find the corresponding starting points GeoDataFrame for the current city
    if city_name in starting_points_key_mapping:
        starting_points_key = starting_points_key_mapping[city_name]
        if (
            starting_points_key in all_starting_points_2019_2024
            and not all_starting_points_2019_2024[starting_points_key].empty
        ):
            starting_point_gdf = all_starting_points_2019_2024[starting_points_key]
            print(f"  Accessed starting points for {city_name} (key: {starting_points_key})")

            # Project CRS if needed
            if starting_point_gdf.crs != G.graph['crs']:
                print(f"  Projecting starting points to graph CRS: {G.graph['crs']}")
                starting_point_gdf = starting_point_gdf.to_crs(G.graph['crs'])

            print(f"  Processing {len(starting_point_gdf)} starting points for {city_name}...")
            isochrones_to_save = []

            # Loop through starting points
            for index, starting_point in starting_point_gdf.iterrows():
                group_id = starting_point.get('group_id', 'unknown_group')

                # Find nearest node in graph
                try:
                    nearest_node_osmid = ox.nearest_nodes(
                        G, starting_point.geometry.x, starting_point.geometry.y
                    )
                except Exception as e:
                    print(f"      Could not find nearest node for start idx {index}, group {group_id}: {e}")
                    continue

                # Loop through travel times
                for travel_time_sec, travel_time_min in zip(travel_times_seconds, travel_times_minutes):
                    try:
                        distances, _ = nx.single_source_dijkstra(
                            G, source=nearest_node_osmid, cutoff=travel_time_sec, weight='travel_time'
                        )
                        reachable_nodes_osmid = set(distances.keys())

                        # Collect reachable edges
                        edges_proj_list = list(G.edges(keys=True, data=True))
                        edge_geoms = [
                            data['geometry'] for u, v, k, data in edges_proj_list
                            if u in reachable_nodes_osmid and v in reachable_nodes_osmid and 'geometry' in data
                        ]

                        if edge_geoms:
                            # Build polygon(s) from edges
                            multiline = MultiLineString(edge_geoms)
                            merged = unary_union(multiline)
                            polygons = list(polygonize(merged))

                            for poly in polygons:
                                isochrones_to_save.append({
                                    'city': city_name,
                                    'group_id': group_id,
                                    'start_pt_idx': index,
                                    'travel_time': travel_time_min,
                                    'geometry': poly
                                })
                        else:
                            print(f"      No edges reachable for {travel_time_min} min (start idx {index}, group {group_id})")

                    except Exception as e:
                        print(f"      Error creating isochrone for {travel_time_min} min (start idx {index}, group {group_id}): {e}")

            # Save Isochrones for city
            if isochrones_to_save:
                isochrone_gdf = gpd.GeoDataFrame(isochrones_to_save, crs=G.graph['crs'])

                safe_city_name = (
                    city_name.replace(" ", "_")
                    .replace("(", "")
                    .replace(")", "")
                    .replace("̈", "e")
                    .replace("ä", "ae")
                    .replace("ö", "oe")
                    .replace("ü", "ue")
                )
                output_filepath = os.path.join(output_dir_service_area, f"{safe_city_name}_isochrones.gpkg")

                try:
                    isochrone_gdf.to_file(output_filepath, driver="GPKG")
                    print(f"  Successfully saved isochrones for {city_name} to {output_filepath}")
                except Exception as e:
                    print(f"  Error saving isochrones for {city_name}: {e}")
            else:
                print(f"  No isochrones to save for {city_name}")

        else:
            print(f"  Starting points not found or empty for {city_name} (key: {starting_points_key})")
    else:
        print(f"  No starting points key found for {city_name}. Skipping.")

print("\nFinished calculating and saving isochrones for all cities.")


Saving isochrones to: /content/drive/MyDrive/Cold Spots Bayern/isochrones_2019_2024
Please enter the city name to start processing with (e.g., 'Nuremberg'): Würzburg

Calculating isochrones for city: Würzburg
  Accessed starting points for Würzburg (key: Wuerzburg_starting_points_2019_2024)
  Processing 2810 starting points for Würzburg...
      No edges reachable for 5 min (start idx 146, group 0.0)
      No edges reachable for 5 min (start idx 152, group 0.0)
      No edges reachable for 5 min (start idx 388, group 0.0)
      No edges reachable for 5 min (start idx 389, group 0.0)
      No edges reachable for 5 min (start idx 390, group 0.0)
      No edges reachable for 5 min (start idx 402, group 0.0)
      No edges reachable for 10 min (start idx 402, group 0.0)
      No edges reachable for 15 min (start idx 402, group 0.0)
      No edges reachable for 20 min (start idx 402, group 0.0)
      No edges reachable for 5 min (start idx 409, group 0.0)
      No edges reachable for 10 min

KeyboardInterrupt: 

hier werden die Straßenabschitte die sich berühren oder überschneiden zusammengefügt und wenn es einen Loop gibt ensteht polygon. allerdongs müssen sie noch zu cold spots gruppiert werden und dann convexe hülle drum herum, damit gesamter erreichbarer raum eingeschlossen wird.